# 🎵 Song Genre Classification using Audio Features (Telugu Dataset)
This notebook trains a neural network to classify Telugu songs by genre. It extracts MFCC features from audio files, trains a model, and outputs accuracy/loss results.

In [ ]:

# 📦 Install dependencies (uncomment if running first time)
# !pip install librosa tensorflow scikit-learn numpy matplotlib


In [ ]:

import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# 📂 Dataset path
DATA_PATH = "Telugu"  # Folder with subfolders for each genre (e.g., Telugu/Folk, Telugu/Melody, etc.)

# 🎧 Extract MFCC features from audio files
def extract_features(file_path, max_pad_len=174):
    try:
        audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast')
        mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        pad_width = max(0, max_pad_len - mfcc.shape[1])
        mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
        return mfcc
    except Exception as e:
        print("❌ Error encountered while parsing file:", file_path)
        return None


In [ ]:

# 🧹 Prepare dataset
features, labels = [], []

for genre in os.listdir(DATA_PATH):
    genre_path = os.path.join(DATA_PATH, genre)
    if not os.path.isdir(genre_path):
        continue
    for file in os.listdir(genre_path):
        file_path = os.path.join(genre_path, file)
        data = extract_features(file_path)
        if data is not None:
            features.append(data)
            labels.append(genre)

# 🧾 Convert to arrays and encode labels
X = np.array(features, dtype=object)
y = np.array(labels)

if len(y) == 0:
    raise ValueError("No labels found! Please check that your dataset folders contain valid audio files.")

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

# 🪓 Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42, shuffle=True)

X_train = np.array([x for x in X_train])
X_test = np.array([x for x in X_test])

# 🧩 Reshape inputs for dense network
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

print("✅ Data prepared successfully!")
print(f"Total samples: {len(X)}")
print(f"Classes: {list(le.classes_)}")


In [ ]:

# 🧠 Build the neural network
model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_categorical.shape[1], activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

# 🚀 Train the model
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_test, y_test), verbose=1)


In [ ]:

# 📊 Evaluate model
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"\n✅ Train Accuracy: {train_acc*100:.2f}%")
print(f"✅ Test Accuracy: {test_acc*100:.2f}%")
print(f"📉 Train Loss: {train_loss:.4f}")
print(f"📉 Test Loss: {test_loss:.4f}")


In [ ]:

# 📈 Plot Accuracy & Loss Curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Test Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Test Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


✅ **Notebook complete!**

Make sure your dataset structure looks like this:
```
Telugu/
 ├── Folk/
 │    ├── song1.wav
 │    ├── song2.wav
 ├── Melody/
 │    ├── song3.wav
 │    ├── song4.wav
```
Then re-run all cells.